# Seçici sinyal · özgün 38-fold bariyer karşılaştırması

**Amaç:** Önümüzdeki 10 bar içinde TP'ye SL'den önce ulaşacak örnekleri seçmek.
TP=1; SL ve timeout=0. Negatif karar kesin SL demek değildir. Aynı mumda çift temas: SL önce.

**Sabit veri:** 1 Mart 2022 04:00–28 Aralık 2025 02:00 UTC, 33.551 saat.
38 fold; train/validation/test bölüm boyutları 5040/1080/720; adım720;
purge24/embargo24; seq64 → 4977/1017/657 dizi. Test bölümü puanlanmaz.
Altı temel özellik, GRU, BCE, wavelet kapalı. Epoch seçimi validation BCE.
B25=2/5, B11=1/1, B12=1/2, B152=1.5/2, B22=2/2, B32=3/2.
Her aday 38 fold × 3 seed = 114 eğitim; toplam **684**. Uzun bir Colab çalışmasıdır.
Adayları ayrı oturumlarda çalıştırabilirsiniz; aynı aday kaldığı fold/epoch'tan devam eder.

**F1 maksimizasyonu ve otomatik kazanan kaldırıldı.** %60/%65/%70 precision
senaryoları seçilmiş politikalar değildir. Precision, recall, sinyal sayısı,
sinyalsiz dönemler, fold/seed istikrarı, SL/timeout ayrımı ve AP/BCE referansları raporlanır.
Aynı validation epoch seçimi için de kullanıldığından sonuçlar iyimser geliştirme sonuçlarıdır.

**Test komutu yoktur.** Eylül/Ekim okunmaz; final eğitim/test ayrı sonraki protokoldür.
A100 (NVIDIA A100-SXM4-40GB) seçin. Drive hesabı: icinbedava34@gmail.com. Eski workspace'i koruyun;
Araştırma motoru advisor-selective-v1.1.0 olarak sabittir. Bu notebook mevcut
selective_a100_v11_001 çalışmasının 17 Ekim aktarım kurtarması için açılır.
RECOVER_EXISTING=True: hazırlık yerine denetim. Önce denetimi okuyun, sonra
ALLOW_REPAIR=True ile 6. hücreden devam edin. Veri/model ayarlarını değiştirmeyin.
Çıktılar/checkpointler Drive'da. Aynı workspace'e eşzamanlı iki runtime yazdırmayın.
İnceleme ZIP'i satır bazlı validation tahminlerini içerir; tam model yedeği tüm workspace'tir.


## 1 — Çalışma ayarları

Aynı deneye devam ederken WORKSPACE_ID ve sürümü değiştirmeyin.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

MODE = 'research'  # recovery: yalnızca tablolar; research: yeni deney; archive: finali oku
PHASE = 'basic'
WORKSPACE_ID = 'selective_a100_v11_001'
EXPERIMENTS = ['B25', 'B11', 'B12', 'B152', 'B22', 'B32']  # İsterseniz tek aday seçin; her aday 114 eğitimdir.
ALLOW_TRAINING = False
RELEASE_TAG = 'advisor-selective-v1.1.0'  # Sabit yayın; dalın güncel ucunu otomatik çekmez.
REPO_URL = 'https://github.com/umutergul74/YeniBotDeney.git'
REPO_DIR = Path('/content/advisor_selective_a100_v11_repo')
RUNTIME_DIR = Path('/content/advisor_selective_a100_v11_runtime')
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot/advisor_experiments')
SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
assert WORKSPACE_ID.replace('_','').replace('-','').isalnum()
assert MODE in ('recovery','research','archive')
assert PHASE != 'final' or MODE == 'archive', 'Görülmüş final testi yeniden çalıştırılmaz.'

os.environ["MPLBACKEND"] = "Agg"
AUTO_DISCONNECT = True
RECOVER_EXISTING = True  # 17 Ekim aktarım kurtarması; mevcut çalışma korunur.
ALLOW_REPAIR = False  # Denetimden sonra True yapın; yalnızca belirlenen eksikler.
SUCCESSFUL_STEPS = {}


## 2 — Drive ve ayrı çıktı alanı

Hazırlanmış tablolar, ağırlıklar ve ilerleme kalıcı alana yazılır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
LEGACY_ROOT = DRIVE_BASE / 'colab_v1'
namespace = {'recovery':'recovery_v2','research':'selective_research_v1','archive':'archive_audit_v2'}[MODE]
ROOT = DRIVE_BASE / namespace / WORKSPACE_ID
ROOT.mkdir(parents=True,exist_ok=True)
LOG_DIR = ROOT / 'sessions' / SESSION
LOG_DIR.mkdir(parents=True,exist_ok=True)
print('Eski arşiv (salt okunur):',LEGACY_ROOT)
print('Bu çalışmanın kalıcı alanı:',ROOT)
print('Oturum günlükleri:',LOG_DIR)


## 3 — Sabit Git yayını

Yerel değişiklik varsa otomatik silinmez; yanlış sürümle devam edilmez.


In [ ]:
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',RELEASE_TAG,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
def git(*args):
    return subprocess.check_output(['git','-C',str(REPO_DIR),*args],text=True).strip()
assert not git('status','--porcelain','--untracked-files=no'), 'Kodda yerel değişiklik var.'
assert git('rev-parse','HEAD') == git('rev-parse',RELEASE_TAG+'^{commit}'), 'Yanlış yayın; yeni runtime açın.'
print('Kod sürümü:',git('rev-parse','HEAD'))

assert git('remote','get-url','origin') == REPO_URL, 'Yanlış depo'


## 4 — Kilitli araştırma ortamı

Sistem Python'una pip install yapılmaz. Ayrı Python + venv kullanılır; bütün
bağımlılıklar sürüm ve SHA ile sabittir. Drive'daki indirme önbelleği tekrar kullanılır.
uv kilitleri ve paket onbellegi /content altinda tutulur. Drive kalici ciktilari
ve dogrulanmis kurucu dosyasini saklar. Yeni runtime paketleri yeniden indirebilir.
Kurulum başarısızsa sonraki hücreye geçmeyin. Eğitim için NVIDIA A100-SXM4-40GB seçin.


In [ ]:
def logged(command, name):
    with (LOG_DIR/name).open('a',encoding='utf-8',buffering=1) as log:
        env=dict(os.environ)
        for key in ('PYTHONPATH','PYTHONHOME','PYTHONSTARTUP','IPYTHONDIR','JUPYTER_CONFIG_DIR'):
            env.pop(key,None)
        env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
        env['MPLBACKEND']='Agg'  # Colab inline backend is host-only.
        with subprocess.Popen(command,cwd=REPO_DIR,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line,end=''); log.write(line)
                code=process.wait()
            except BaseException:
                process.terminate(); process.wait(); raise
            if code: raise subprocess.CalledProcessError(code,command)
RECEIPT=RUNTIME_DIR/'environment_receipt.json'
logged([sys.executable,'-I',str(REPO_DIR/'scripts/bootstrap_advisor.py'),
        '--repo',str(REPO_DIR),'--runtime',str(RUNTIME_DIR),
        '--cache',str(DRIVE_BASE/'reproducibility_cache_v2'),
        '--receipt',str(RECEIPT)],'environment_setup.log')
runtime=json.loads(RECEIPT.read_text())
RESEARCH_PYTHON=runtime['python']
print('İzole Python:',RESEARCH_PYTHON)
print('Paket kilidi:',runtime['lock_sha256'])


def step_identity():
    return (str(ROOT), RELEASE_TAG, tuple(EXPERIMENTS), ALLOW_TRAINING, RECOVER_EXISTING, ALLOW_REPAIR)

def execute_step(action):
    order = ['prepare', 'train', 'report', 'bundle']
    for stale in order[order.index(action):]:
        SUCCESSFUL_STEPS.pop(stale, None)
    if RECOVER_EXISTING:
        if tuple(EXPERIMENTS) != ('B25','B11','B12','B152','B22','B32'):
            raise ValueError('Recovery requires the six original candidates')
        mapped = dict(prepare='audit', train='repair', report='report', bundle='bundle')[action]
        if mapped == 'repair' and not ALLOW_REPAIR:
            raise RuntimeError('ALLOW_REPAIR=False; audit only')
        command = [RESEARCH_PYTHON, '-I', str(RECOVERY_HELPER), mapped, '--repo', str(REPO_DIR),
                   '--root', str(ROOT), '--session', SESSION]
        if ALLOW_REPAIR: command.append('--allow-repair')
        logged(command, 'recovery_'+mapped+'.log')
        SUCCESSFUL_STEPS[action] = step_identity()
        return
    command = [RESEARCH_PYTHON, '-I', str(REPO_DIR/'scripts/advisor_selective.py'), action,
               '--root', str(ROOT), '--legacy', str(LEGACY_ROOT), '--session', SESSION,
               '--candidates', *EXPERIMENTS]
    if ALLOW_TRAINING: command.append('--allow-training')
    logged(command, action+'.log')
    SUCCESSFUL_STEPS[action] = step_identity()

# Host-only launcher; research imports remain in the original pinned interpreter.
import urllib.request, hashlib
RECOVERY_HELPER = RUNTIME_DIR/'advisor_recover_selective.py'
helper_bytes = urllib.request.urlopen('https://raw.githubusercontent.com/umutergul74/YeniBotDeney/advisor-selective-recovery-v1.0.0/scripts/advisor_recover_selective.py').read()
assert hashlib.sha256(helper_bytes).hexdigest() == 'ece3edbcc66f45664c6a132452abb1c761d60bea2534f557ef2dbe04b123a3e2', 'Recovery helper hash mismatch'
RECOVERY_HELPER.write_bytes(helper_bytes)

def sync_verified():
    # A fresh mount must contain the same bytes before runtime release.
    import hashlib
    from google.colab import drive
    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(4*1024*1024), b''):
                h.update(block)
        return h.hexdigest()
    expected = {str(p.relative_to(ROOT)): digest(p) for p in ROOT.rglob('*') if p.is_file()}
    if not expected: raise RuntimeError('Empty workspace; refusing shutdown')
    print('Drive aktarımı ve yeniden bağlanarak dosya doğrulaması başlıyor.', flush=True)
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    for relative, checksum in expected.items():
        p = ROOT/relative
        if not p.is_file() or digest(p) != checksum:
            raise RuntimeError('Buluttaki dosya eksik/farklı; oturum açık bırakıldı: '+relative)
    print('Drive yeniden bağlandı: bütün dosya hashleri eşleşti.', flush=True)


## 5 · Sabit kohort / kurtarma denetimi

RECOVER_EXISTING=True ise yeni veri hazırlamaz; 684 foldun içeriklerini denetler. Ham snapshot hashleri, 33.551 ortak saat, özellikler ve 38 fold doğrulanır. Yeni veri indirilmez.


In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'
execute_step('prepare')


## 6 · Eksikleri tamamlama / eğitim

Kurtarma modunda 34 korunur (ağırlığı eksik), 35 geçmişi checkpointten çıkarılır, 36 sürdürülür, 37 yeniden eğitilir. Mevcut dosyalar ezilmez. Normal eğitim için ALLOW_TRAINING=True seçin. Her aday 114 eğitim; checkpointler kalıcı. Aday listesi daraltılabilir.


In [ ]:
if ALLOW_TRAINING or (RECOVER_EXISTING and ALLOW_REPAIR):
    execute_step('train')
else:
    print('Eğitim kapalı; ayarlarda ALLOW_TRAINING=True seçin.')


## 7 · Seçicilik raporu

Seçili adayların 38×3 koşusu tamamlanmalıdır. Eşik veya kazanan otomatik seçilmez. Precision hedefleri betimseldir.


In [ ]:
if ALLOW_TRAINING or RECOVER_EXISTING:
    execute_step('report')
else:
    print('Rapor eğitim tamamlandıktan sonra hazırlanır.')


## 8 · Ayrıntılı inceleme paketi

Validation tahminleri, history ve metrikler ZIP içinde. Ham veri/ağırlıklar için tüm workspace yedeklenir. Tüm seçili adayların eğitim, rapor ve ZIP adımları başarıyla tamamlanınca runtime otomatik bırakılır (AUTO_DISCONNECT=True). Hata varsa kapatılmaz.


In [ ]:
execute_step('bundle')
print('Raporlar:', ROOT/'reports')
print('Modeller:', ROOT/'runs')
if AUTO_DISCONNECT and (ALLOW_TRAINING or (RECOVER_EXISTING and ALLOW_REPAIR)) and all(SUCCESSFUL_STEPS.get(step) == step_identity() for step in ('prepare', 'train', 'report', 'bundle')):
    sync_verified()
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    runtime.unassign()
else:
    print('Otomatik kapatma yapılmadı: bütün aşamalar bu ayarlarla başarıyla tamamlanmalı.')
